# 02. Limpieza y Enriquecimiento de Datos (Feature Engineering)

Una vez que entendieron los datos, vamos a prepararlos para que los algoritmos de Machine Learning puedan procesarlos correctamente. En la vida real, lo mejor es empaquetar esto automatizado en Scikit-Learn Pipelines o en scripts de Python (`src/features/build_features.py`). Aquí puedes experimentar con las rutinas de limpieza.

### Instrucciones Generales:
1. **Solvertar problema de calidad:**: Solucionar problema de calidad encontrados en el EDA: consistencia, sensibilidad, precision y completitud. Documenta cada decision tomada.
2. **Codificación Categórica:** El campo `ocean_proximity` es de texto. Conviértelo en variable numerica, ya que los algoritmos clasicos no entienen el texto. Documenta porque usaste codificacion Ordinal o Nominal.
3. **Enriquecimiento (Feature Engineering):** Como pudiste notar en tu análisis, `total_rooms` no significa mucho si hay muchos hogares en un distrito. Agrega nuevas métricas útiles, por ejemplo:
   - `rooms_per_household = total_rooms / households`
   - `bedrooms_per_room = total_bedrooms / total_rooms`
   - `population_per_household = population / households`
4. **Escalado de Variables:** Aplica un `StandardScaler` o `MinMaxScaler` para evitar que las variables numéricas grandes pesen más en algoritmos basados en distancias o gradientes.


In [1]:
# Escribe tu código aquí para limpiar y enriquecer los datos
import pandas as pd
import numpy as np


In [9]:
raw_data = pd.read_csv(r'../data/interim/train_set.csv')

In [10]:
raw_data.describe()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,16512.000000,16512.000000,16512.000000,16512.000000,16344.000000,16512.000000,16512.000000,16512.000000,16512.000000
mean,-119.573125,35.637746,28.577156,2639.402798,538.949094,1425.513929,499.990189,3.870428,206333.518653
std,2.000624,2.133294,12.585738,2185.287466,423.862079,1094.795467,382.865787,1.891936,115314.047529
min,-124.350000,32.550000,1.000000,2.000000,1.000000,3.000000,1.000000,0.499900,14999.000000
25%,-121.800000,33.930000,18.000000,1447.000000,296.000000,787.000000,279.000000,2.562500,119200.000000
50%,-118.510000,34.260000,29.000000,2125.000000,434.000000,1167.000000,408.000000,3.538500,179200.000000
75%,-118.010000,37.720000,37.000000,3154.000000,645.000000,1726.000000,603.000000,4.750000,263925.000000
max,-114.490000,41.950000,52.000000,39320.000000,6210.000000,16305.000000,5358.000000,15.000100,500001.000000


### Problemas de calidad

In [11]:
#1. Duplicados
#__________Explicitos________
clean_data = raw_data.drop_duplicates(keep='last')
borrados = len(raw_data) - len(clean_data)
print(f"--- Reporte de Duplicados Explicitos ---")
print(f"Se eliminaron {borrados} registros duplicados.")
print(f"Registros restantes: {len(clean_data)}")

#__________Logicos____________
clean_data['income_rounded'] = clean_data['median_income'].round(2)
columnas_logicas = ['longitude', 'latitude', 'housing_median_age', 'income_rounded']
total_antes = len(clean_data)
clean_data.drop_duplicates(subset=columnas_logicas, keep='last', inplace=True)
clean_data.drop(columns=['income_rounded'], inplace=True)
eliminados = total_antes - len(clean_data)
print(f"--- Reporte de Duplicados Lógicos ---")
print(f"Se eliminaron registros con valores iguales de ubicacion, antiguedad e ingresos: {eliminados}")
print(f"Registros restantes: {len(clean_data)}")

--- Reporte de Duplicados Explicitos ---
Se eliminaron 0 registros duplicados.
Registros restantes: 16512
--- Reporte de Duplicados Lógicos ---
Se eliminaron registros con valores iguales de ubicacion, antiguedad e ingresos: 5
Registros restantes: 16507


In [12]:
#2. Valores ausentes
total_filas = len(clean_data)
filas_con_nulos = clean_data.isnull().any(axis=1).sum()
porcentaje_nulos = (filas_con_nulos / total_filas) * 100

print(f"Porcentaje de filas con nulos: {porcentaje_nulos:.2f}%")

#______ Eliminar nulos si el porcentaje es menor al 3%________
if porcentaje_nulos < 3:
    clean_data.dropna(inplace=True)
    print(f"Registros eliminados: {filas_con_nulos}")
#______ Imputar nulos con mediana si es un porcentaje mayor_____
else:
    columnas_numericas = clean_data.select_dtypes(include=['number']).columns
    for col in columnas_numericas:
        mediana = clean_data[col].median()
        clean_data[col].fillna(mediana, inplace=True)
    
    print(f"Se imputaron nulos con la mediana.")

# Resultado final
print(f"Registros finales en clean_data: {len(clean_data)}")
print(f"Nulos restantes: {clean_data.isnull().sum().sum()}")

Porcentaje de filas con nulos: 1.01%
Registros eliminados: 167
Registros finales en clean_data: 16340
Nulos restantes: 0


In [13]:
#3. Outliers
# __________LIMITES, eliminar registros con techos artificiales_____________
filas_antes = len(clean_data)
clean_data = clean_data[clean_data['median_house_value'] < 500001]
filas_eliminadas = filas_antes - len(clean_data)
porcentaje_eliminado = (filas_eliminadas / filas_antes) * 100
print(f"Filas eliminadas: {filas_eliminadas}")
print(f"Porcentaje eliminado: {porcentaje_eliminado:.2f}%")

# Outliers, rango intercuartil
columnas_iqr = [
    'total_rooms',
    'total_bedrooms',
    'population',
    'households'
]

# Guardar cantidad original
filas_antes = len(clean_data)
# Eliminar outliers directamente en clean_data
for col in columnas_iqr:
    Q1 = clean_data[col].quantile(0.25)
    Q3 = clean_data[col].quantile(0.75)
    IQR = Q3 - Q1

    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    clean_data = clean_data[
        (clean_data[col] >= limite_inferior) &
        (clean_data[col] <= limite_superior)
    ]

clean_data.reset_index(drop=True, inplace=True)
filas_eliminadas = filas_antes - len(clean_data)
print("Filas originales:", filas_antes)
print("Filas eliminadas:", filas_eliminadas)
print("Filas restantes:", len(clean_data))

#TRANSFORMACIÓN LOG para disminuir la cola 
clean_data['median_income_log'] = np.log1p(clean_data['median_income'])


# WINSORIZAR columnas con colas largas (Population y Bedrooms)
#for col in ['population', 'total_bedrooms','total_rooms','households']:
#    limite = clean_data[col].quantile(0.99)
#    clean_data[col] = clean_data[col].clip(upper=limite)

#TRANSFORMACIÓN LOG para disminuir la cola y que la distribucion se parezca mas a una "normal"
#clean_data['median_income_log'] = np.log1p(clean_data['median_income'])
#clean_data['population_log'] = np.log1p(clean_data['population'])
#clean_data['total_bedrooms_log'] = np.log1p(clean_data['total_bedrooms'])
#clean_data['total_rooms_log'] = np.log1p(clean_data['total_rooms'])
#clean_data['households_log'] = np.log1p(clean_data['households'])




Filas eliminadas: 758
Porcentaje eliminado: 4.64%
Filas originales: 15582
Filas eliminadas: 1912
Filas restantes: 13670


### Codificación Categorica

In [14]:
# 1. Proximidad al oceano
# Ordenamos de lo más alejado a lo más cerca
ocean_mapping = {
    'INLAND': 0,
    '<1H OCEAN': 1,
    'NEAR OCEAN': 2,
    'NEAR BAY': 3,
    'ISLAND': 4
}

clean_data['ocean_ordinal'] = clean_data['ocean_proximity'].map(ocean_mapping)

# 2. Antiguedad
bins_age = [0, 18, 35, np.inf]
labels_age = ['Nueva', 'Media', 'Vieja']
age_mapping = {'Nueva': 0, 'Media': 1, 'Vieja': 2}

clean_data['age_ordinal'] = pd.cut(clean_data['housing_median_age'], 
                                   bins=bins_age, 
                                   labels=labels_age).map(age_mapping).astype(int)

# Eliminar las columnas originales para evitar que el modelo reciba datos redundantes
cols_to_drop = ['ocean_proximity', 'housing_median_age']
clean_data.drop(columns=cols_to_drop, inplace=True)

clean_data[['ocean_ordinal', 'age_ordinal']].sample(10)

,ocean_ordinal,age_ordinal
11344,1,2
10385,0,0
11873,2,1
10440,1,1
9663,1,0
756,0,1
3323,0,0
12232,0,1
528,0,1
13117,1,1


### Enriquecimiento

In [15]:
# 1. Habitaciones por Hogar
clean_data["rooms_per_household"] = (clean_data["total_rooms"] / clean_data["households"]).round(2)

In [16]:
# 2. Dormitorios por Habitación
clean_data["bedrooms_per_room"] = (clean_data["total_bedrooms"] / clean_data["total_rooms"]).round(4)

In [17]:
# 3. Población por Hogar
clean_data['population_per_household'] = (clean_data['population'] / clean_data['households']).round(2)

In [18]:
# 4. Ratio de Ingreso sobre Valor de la Zona
clean_data["income_per_person"] = (clean_data["median_income"] / clean_data["population_per_household"]).round(4)
# Ajusta el ingreso mediano según cuántas personas dependen de ese ingreso en promedio dentro del hogar.

In [19]:
correlaciones = clean_data.corr(numeric_only=True)["median_house_value"].sort_values(ascending=False)
print(correlaciones)

median_house_value          1.000000
income_per_person           0.702682
median_income               0.652626
median_income_log           0.636585
ocean_ordinal               0.402356
total_rooms                 0.203886
households                  0.107279
rooms_per_household         0.107277
total_bedrooms              0.075014
age_ordinal                 0.062679
population                 -0.022920
longitude                  -0.036118
latitude                   -0.158752
population_per_household   -0.178487
bedrooms_per_room          -0.235240
Name: median_house_value, dtype: float64


In [20]:
clean_data.drop(columns=['median_income'], inplace=True)

In [21]:
clean_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13670 entries, 0 to 13669
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   longitude                 13670 non-null  float64
 1   latitude                  13670 non-null  float64
 2   total_rooms               13670 non-null  float64
 3   total_bedrooms            13670 non-null  float64
 4   population                13670 non-null  float64
 5   households                13670 non-null  float64
 6   median_house_value        13670 non-null  float64
 7   median_income_log         13670 non-null  float64
 8   ocean_ordinal             13670 non-null  int64  
 9   age_ordinal               13670 non-null  int64  
 10  rooms_per_household       13670 non-null  float64
 11  bedrooms_per_room         13670 non-null  float64
 12  population_per_household  13670 non-null  float64
 13  income_per_person         13670 non-null  float64
dtypes: flo

In [22]:
clean_data.describe()

,longitude,latitude,total_rooms,total_bedrooms,population,households,median_house_value,median_income_log,ocean_ordinal,age_ordinal,rooms_per_household,bedrooms_per_room,population_per_household,income_per_person
count,13670.000000,13670.000000,13670.000000,13670.000000,13670.000000,13670.000000,13670.000000,13670.000000,13670.000000,13670.000000,13670.00000,13670.000000,13670.000000,13670.000000
mean,-119.613619,35.711342,2080.028456,425.530358,1147.382882,398.412143,189885.515435,1.484098,1.002853,1.098830,5.37588,0.213164,2.963623,1.315920
std,2.001957,2.166649,1026.400800,199.710611,549.163389,185.323077,98272.671800,0.333718,0.948637,0.741794,2.33353,0.055069,1.078400,0.627353
min,-124.350000,32.550000,2.000000,2.000000,3.000000,2.000000,14999.000000,0.405398,0.000000,0.000000,0.89000,0.100000,0.690000,0.025500
25%,-121.790000,33.940000,1353.000000,282.000000,753.000000,267.000000,112500.000000,1.250790,0.000000,1.000000,4.46000,0.177400,2.460000,0.839975
50%,-118.700000,34.410000,1937.000000,400.000000,1082.000000,378.000000,170800.000000,1.491319,1.000000,1.000000,5.21000,0.203600,2.840000,1.249200
75%,-118.020000,37.750000,2697.750000,553.000000,1502.000000,518.000000,244400.000000,1.718579,1.000000,2.000000,5.97000,0.238000,3.300000,1.689650
max,-114.490000,41.950000,5650.000000,1047.000000,2747.000000,905.000000,500000.000000,2.772595,4.000000,2.000000,132.53000,1.000000,63.750000,6.756800


### Escalado: Realizado en experimentacion

In [ ]:
clean_data.to_csv('clean_data.csv', index=False)

#### Otro enfoque no funcional

In [ ]:
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

features_geo = ['latitude', 'longitude']
# Columnas con posibles outliers (valores extremos)
features_robust = [
    'income_per_person', 
    'rooms_per_household', 
    'bedrooms_per_room', 
    'population_per_household',
    'population',
    'total_bedrooms',
    'total_rooms',
    'households'   
]
# Columnas para estandarización normal
features_std = [
    'median_income_log',
]

# 2. Configurar el ColumnTransformer
# Nota: 'ocean_ordinal' y 'age_ordinal' pasarán sin cambios (passthrough)
ct = ColumnTransformer([
    ('geo', MinMaxScaler(), features_geo),
    ('robust', RobustScaler(), features_robust),
    ('std', StandardScaler(), features_std)
], remainder='passthrough')

# 3. Dividir ANTES de escalar (Para evitar Data Leakage)
# Asumiendo que 'median_house_value' es tu objetivo (Y)
X = clean_data.drop('median_house_value', axis=1)
y = clean_data['median_house_value']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Ajustar y Transformar
X_train_scaled = ct.fit_transform(X_train)
X_val_scaled = ct.transform(X_val)

# 5. Guardar para el otro notebook
joblib.dump(ct, 'scaler_housing.joblib')
# También puedes guardar los arrays resultantes si prefieres

# Guardar csv
target = 'median_house_value'
train_final = pd.DataFrame(X_train_scaled, columns=X.columns)
train_final[target] = y_train.values

valid_final = pd.DataFrame(X_val_scaled, columns=X.columns)
valid_final[target] = y_val.values

train_final.to_csv('data_train_ML.csv', index=False)
valid_final.to_csv('data_valid_ML.csv', index=False)